In [36]:
import pandas as pd 
import numpy as np 

In [37]:
df= pd.read_csv("../data/raw/github_top_repositories_V2.csv")

In [38]:
df.head(2)

,Domain,Repository Name,Full Name,Description,Primary Language,Stars Count,Forks Count,Open Issues Count,Has Wiki,Has Pages,Has Projects,Size (KB),Created At,Updated At,Pushed At,Default Branch,Owner Login,Owner Type,License,Topics
0,Machine Learning,tensorflow,tensorflow/tensorflow,An Open Source Machine Learning Framework for ...,C++,194622,75263,4302,False,False,True,1302841,2015-11-07T01:19:20Z,2026-04-10T10:08:00Z,2026-04-10T10:13:20Z,master,tensorflow,Organization,Apache License 2.0,"deep-learning, deep-neural-networks, distribut..."
1,Machine Learning,transformers,huggingface/transformers,🤗 Transformers: the model-definition framework...,Python,159148,32816,2365,True,False,True,463713,2018-10-29T13:56:00Z,2026-04-10T10:08:23Z,2026-04-10T10:08:12Z,main,huggingface,Organization,Apache License 2.0,"audio, deep-learning, deepseek, gemma, glm, ha..."


**EDA**

In [39]:
df.shape

(5000, 20)

In [40]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 20 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   Domain             5000 non-null   object
 1   Repository Name    5000 non-null   object
 2   Full Name          5000 non-null   object
 3   Description        5000 non-null   object
 4   Primary Language   5000 non-null   object
 5   Stars Count        5000 non-null   int64 
 6   Forks Count        5000 non-null   int64 
 7   Open Issues Count  5000 non-null   int64 
 8   Has Wiki           5000 non-null   bool  
 9   Has Pages          5000 non-null   bool  
 10  Has Projects       5000 non-null   bool  
 11  Size (KB)          5000 non-null   int64 
 12  Created At         5000 non-null   object
 13  Updated At         5000 non-null   object
 14  Pushed At          5000 non-null   object
 15  Default Branch     5000 non-null   object
 16  Owner Login        5000 non-null   object


In [41]:
# Missing Values

df.isna().sum()

Domain               0
Repository Name      0
Full Name            0
Description          0
Primary Language     0
Stars Count          0
Forks Count          0
Open Issues Count    0
Has Wiki             0
Has Pages            0
Has Projects         0
Size (KB)            0
Created At           0
Updated At           0
Pushed At            0
Default Branch       0
Owner Login          0
Owner Type           0
License              0
Topics               0
dtype: int64

In [42]:
# Checking Duplicates

df.duplicated().sum()

np.int64(3)

In [43]:
# Most common languages

df['Primary Language'].value_counts().head(10)

Primary Language
Python              1086
0                    602
C++                  486
JavaScript           462
Go                   436
Java                 370
Rust                 323
TypeScript           278
Jupyter Notebook     224
C                    107
Name: count, dtype: int64

*Python is the most commonly used language*

In [44]:
# Most common domains

df['Domain'].value_counts()

Domain
Machine Learning               200
Cybersecurity                  200
Robotics                       200
Internet of Things             200
Computer Vision                200
Natural Language Processing    200
Artificial Intelligence        200
Cloud Computing                200
Game Development               200
Backend Development            200
Frontend Development           200
DevOps                         200
Blockchain                     200
Deep Learning                  200
iOS                            200
Android                        200
Web Development                200
Data Science                   200
Rust                           200
Go                             200
C++                            200
Java                           200
JavaScript                     200
Python                         200
Software Engineering           200
Name: count, dtype: int64

*Different domains are balanced in the dataset*

In [45]:
df.loc[33,'Topics']

'angular, archiving, django, dms, document-management, document-management-system, hacktoberfest, machine-learning, ocr, optical-character-recognition, pdf'

In [46]:
# Most frequent topics

df['Topics'].dropna().str.split(",").explode().str.split().value_counts().head(15)

Topics
[machine-learning]           882
[deep-learning]              789
[python]                     728
[javascript]                 442
[android]                    411
[hacktoberfest]              401
[data-science]               397
[computer-vision]            388
[artificial-intelligence]    383
[ai]                         369
[ios]                        314
[nlp]                        313
[pytorch]                    312
[frontend]                   307
[devops]                     290
Name: count, dtype: int64

In [47]:
# Missing Descriptions

(df['Description'].str.strip() == "").sum() # Count of empty entries after removing spaces

np.int64(0)

*No missing descriptions*

In [48]:
# Checking description to determine preprocessing

df['Description'].sample(10)


3124    前端开发博客，分享互联网最精彩的前端技术，欢迎关注我微信公众号：前端开发博客，回复 1024...
2664    🎓 Because Education should be free. Contributi...
4692                             Tracking Any Point (TAP)
1077    The official GitHub mirror of the Chromium source
3831    Production-grade Rust-native trading engine wi...
2581    Roadmap of learning blockchain technology and ...
1716                  🌊 Online machine learning in Python
4180    👩‍🏫 Advanced NLP with spaCy: A free online course
4543    This is an IoT device communication protocol i...
4020    :book: A curated list of resources dedicated t...
Name: Description, dtype: object

In [49]:
df.loc[37,'Description']

'吴恩达老师的机器学习课程个人笔记'

**PREPROCESSING**

**removing unnecessary columns**

In [51]:
cols = [
    "Full Name",
    "Repository Name",
    "Description",
    "Topics",
    "Domain",
    "Primary Language",
    "Stars Count",
    "Forks Count",
    "Updated At",
]

df = df[cols].copy()

In [52]:
df.shape

(5000, 9)

**Handling Duplicates**

In [53]:
df.drop_duplicates(inplace=True)

In [54]:
print(df.columns)

Index(['Full Name', 'Repository Name', 'Description', 'Topics', 'Domain',
       'Primary Language', 'Stars Count', 'Forks Count', 'Updated At'],
      dtype='object')


**Checking Null**

In [55]:
df.isnull().sum()

Full Name           0
Repository Name     0
Description         0
Topics              0
Domain              0
Primary Language    0
Stars Count         0
Forks Count         0
Updated At          0
dtype: int64

In [56]:
for col in ["Description", "Topics", "Domain", "Primary Language"]:
    print(col, (df[col].astype(str).str.strip() == "0").sum())

Description 11
Topics 180
Domain 0
Primary Language 602


In [57]:
text_cols = ["Description", "Topics", "Domain", "Primary Language"]

for col in text_cols:
    df[col] = df[col].astype(str).replace("0", "")

**Selecting unique identification feature**

In [58]:
df['Full Name'].value_counts().head(20)

Full Name
ashishpatel26/500-AI-Machine-learning-Deep-learning-Computer-vision-NLP-Projects-with-code    6
explosion/spaCy                                                                               6
microsoft/AI-For-Beginners                                                                    5
voxel51/fiftyone                                                                              5
Tencent/ncnn                                                                                  5
bharathgs/Awesome-pytorch-list                                                                5
huggingface/datasets                                                                          5
google-ai-edge/mediapipe                                                                      5
kornia/kornia                                                                                 5
Developer-Y/cs-video-courses                                                                  5
AMAI-GmbH/AI-Expert-Roadmap   

In [59]:
df[df["Full Name"] == "explosion/spaCy"][
    ["Full Name", "Repository Name", "Domain", "Primary Language"]
]

,Full Name,Repository Name,Domain,Primary Language
43,explosion/spaCy,spaCy,Machine Learning,Python
234,explosion/spaCy,spaCy,Deep Learning,Python
578,explosion/spaCy,spaCy,Python,Python
1612,explosion/spaCy,spaCy,Data Science,Python
3812,explosion/spaCy,spaCy,Artificial Intelligence,Python
4008,explosion/spaCy,spaCy,Natural Language Processing,Python


In [ ]:
# What columns vary across duplicate full names

check_cols = [
    "Repository Name",
    "Full Name",
    "Description",
    "Topics",
    "Primary Language",
    "Stars Count",
    "Forks Count",
    "Updated At"
]

duplicate_groups = df.groupby("Full Name")[check_cols].nunique()

problematic = duplicate_groups[
    duplicate_groups.max(axis=1) > 1
]

len(problematic)

38

In [62]:
variation_counts = (
    duplicate_groups.loc[problematic.index] > 1
).sum()

variation_counts.sort_values(ascending=False)

Updated At          34
Stars Count         28
Forks Count         10
Repository Name      0
Full Name            0
Description          0
Topics               0
Primary Language     0
dtype: int64

In [65]:
df_clean = df.copy()

In [67]:
# Sorting dataframe by Updated At

df_clean['Updated At'] = pd.to_datetime(df_clean['Updated At'])

df_clean = df_clean.sort_values("Updated At")

In [ ]:
# separate table with full name and combiined domains

domains = (
    df_clean.groupby("Full Name")["Domain"]
    .apply(lambda x: " ".join(x.unique()))
    .rename("Domain")
)


df_clean = (
    df_clean
    .drop(columns="Domain")
    .drop_duplicates("Full Name", keep="last")  # Keep latest row
    .merge(domains, on="Full Name", how="left")
)

In [70]:
df_clean.shape

(4253, 9)

In [71]:
df_clean["Full Name"].nunique()

4253

**Feature Engineering**

In [74]:
df_clean['combined_text']= (
    df_clean['Description'] + ' ' +
    df_clean['Topics'] + ' ' +
    df_clean['Domain'] 
)

In [76]:
df_clean.head(1)

,Full Name,Repository Name,Description,Topics,Primary Language,Stars Count,Forks Count,Updated At,Domain,combined_text
0,robertpeteuil/multi-cloud-control,multi-cloud-control,Multi cloud control of VM Instances across AWS...,"alibaba-cloud, alibaba-cloud-cli, alibabacloud...",Python,35,10,2024-03-14 22:40:50+00:00,Cloud Computing,Multi cloud control of VM Instances across AWS...


**preprocessing text**

In [77]:
import re
import string

def preprocess(text):
    text= text.lower()
    text= re.sub(r"http\S+\www\S+", "", text)   # urls
    text= text.translate(str.maketrans("","", string.punctuation))  #punctuations
    text= re.sub(r"\s+", " ", text).strip() #extra spaces

    return text

df_clean['combined_text'] = df_clean['combined_text'].apply(preprocess)

In [78]:
df_clean.head(1)

,Full Name,Repository Name,Description,Topics,Primary Language,Stars Count,Forks Count,Updated At,Domain,combined_text
0,robertpeteuil/multi-cloud-control,multi-cloud-control,Multi cloud control of VM Instances across AWS...,"alibaba-cloud, alibaba-cloud-cli, alibabacloud...",Python,35,10,2024-03-14 22:40:50+00:00,Cloud Computing,multi cloud control of vm instances across aws...


In [79]:
df_clean.loc[0,'combined_text']

'multi cloud control of vm instances across aws azure gcp and alicloud unified instance management alibabacloud alibabacloudcli alibabacloud aws awsec2 azure azurevm cliutility cloudcomputing cloudmanagement gcp gevent googlecloudcompute googlecloudplatform libcloud multicloud multiprovider pythonmultiprocessing cloud computing'

In [80]:
df_clean.to_csv("../data/processed/repositories.csv", index = False)